In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import os
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [5]:
####### load common directories FOR APPLIED CONFOCAL PCA
time_interval = 5 #sec/frame
whichpcs = [1,7]
basedir = 'E:/Aaron/Combined_37C_Confocal_PCA_s5_LLS_Apply/'
datadir = basedir + 'Data_and_Figs/'
FullFrame = pd.read_csv(datadir + 'All_Data_with_CGPS_bins.csv', index_col=0)
centers = pd.read_csv(datadir+'PC_bin_centers.csv', index_col=0)
nbins = len(centers.iloc[:,0])
ttot = 3600
ntrans = 2
bsiter = 3000

In [3]:
### restrict data to RANDOM
treatments = ['Random']

savedir = basedir + 'random/'
if not os.path.exists(savedir):
    os.makedirs(savedir)

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment=='Random'].copy()

In [6]:
if __name__ ==  '__main__':
#     ########### get raw transitions and pairs ###########
#     rawtrans = DetailedBalance.get_raw_cgps_trajectories(
#             TotalFrame, #pandas dataframe with all of the cgps binned data
#             whichpcs, #which two PCs to use in the cgps [x,y]
#             time_interval, #real time between datapoints
#             savedir, #where to save the aggregated trajectories
#             )


#     ########### interpolate all transitions so that only individual transitions are made ###########
#     transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
#             TotalFrame, #pandas dataframe with all of the cgps binned data
#             whichpcs, #which two PCs to use in the cgps [x,y]
#             time_interval, #real time between datapoints
#             savedir, #where to save the aggregated trajectories
#             )
    
    
#     ############## get the counts of cells leaving 
#     trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
#             transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
#             whichpcs, #which two PCs to use in the cgps [x,y]
#             savedir, #where to save the aggregated counts
#             nbins, #how many bins in the x and y cgps axes
#             )

    ############## BOOTSTRAP MANY TRAJECTORIES ##########
    bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
            rawtrans, #raw transition pairs from get_raw_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            time_interval, #real time between datapoints
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            ttot, #set the total bootstrap time
            ntrans, #how many transitions to sample at each step
            bsiter, #number of times to bootstrap
            )


    ############# open average bootstrapped currents ###################
    bsfield_sep = DetailedBalance.get_avg_current_error(
            bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            ntrans, #how many transitions to sample at each step
            )
    

Boostrapping trajectories with 2 transition samples for Random


100%|██████████| 3000/3000 [01:47<00:00, 27.98it/s]


Interpolating trajectories for Random
Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [02:00<00:00, 24.88it/s]


Finished bootstrapping


In [6]:
################ get aer and cfs ##################
if __name__ ==  '__main__':
    if os.path.exists(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv'):
        bstrans = pd.read_csv(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv', index_col=0)
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
        center = [12,11]
        DetailedBalance.get_aer_cf(
            bstrans, #boostrapped transitions from get_bootstrapped_cgps_trajectories
            nbins, #how many bins in the x and y cgps axes
            xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
            center, #origin in [x bin,y bin]
            savedir, #where to save calculated aers and cfs
            whichpcs, #which two PCs to use in the cgps [x,y]
            ntrans,
            )

100%|██████████| 3000/3000 [00:19<00:00, 154.54it/s]


In [12]:
########## get individual cell actual aer and cfs ###############

# define the center of the cycle to calculate aer around
center = [12,11] 
#get the area scaling in x and y based on the size of the bins in the cgps
xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

#open the raw transitions in case I didn't just generate them
rawtrans = pd.read_csv(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv', index_col = 0)
#add a movie columns to separate dataframe on
rawtrans['Movie'] = [x.split('_frame')[0] for x in rawtrans.cell.to_list()]
#set the origin to the actual center
results = []
for i, cells in rawtrans.groupby('CellID'):
    movielist = sorted(cells.Movie.unique(),key = lambda x: int(x.split('-')[-3]))
    for m in movielist:
        curmov = cells[cells.Movie == m]
        curmov, runs = utils.get_consecutive_timepoints(curmov, 'frame',1)
        for r in runs:
            cell = curmov.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                cell,
                nbins,
                xyscaling,
                center,
                )))

#make a dataframe and save it
allaers = pd.concat(results, ignore_index=True)
justaers = allaers[['CellID','cell','Treatment','Movie','frame','aer','angular_velocity']].copy()
justaers.to_csv(savedir + f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv')
